# GemCol Evaluation: Phase 3 (BGE Dense Exact Baseline)
This notebook encodes the corpus using `BAAI/bge-large-en-v1.5` and performs chunked exact search using PyTorch matrix multiplication. This bypasses Qdrant Local completely, preventing SQLite deadlocks and OOM crashes, while delivering perfect cosine similarities.

In [ ]:
import sys
import os
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

sys.path.insert(0, '/workspace/gemcol_evaluation')
from utils import load_msmarco_dev, save_run_json, evaluate_run, print_metrics, LatencyProfiler, ExperimentTracker, sanity_check_ndcg

print("Loading MS MARCO dataset...")
queries, qrels, corpus = load_msmarco_dev()
doc_ids = list(corpus.keys())
passages = list(corpus.values())

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading BGE model on {device}...")
model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

query_ids = list(queries.keys())
# BGE requires a special instruction prepended to queries for retrieval tasks
query_texts = [f"Represent this sentence for searching relevant passages: {q}" for q in queries.values()]

print("Encoding queries...")
query_embeddings = model.encode(query_texts, batch_size=256, show_progress_bar=True, normalize_embeddings=True, convert_to_tensor=True)
query_embeddings = query_embeddings.to(device)

In [ ]:
k = 100
num_queries = len(query_ids)
global_top_scores = torch.full((num_queries, k), -float('inf'), device=device)
global_top_indices = torch.full((num_queries, k), -1, dtype=torch.long, device=device)

checkpoint_path = "/workspace/results/bge_exact_checkpoint.pt"
start_idx = 0

if os.path.exists(checkpoint_path):
    print("Found checkpoint! Resuming...")
    checkpoint = torch.load(checkpoint_path)
    start_idx = checkpoint['last_chunk_end']
    global_top_scores = checkpoint['global_top_scores'].to(device)
    global_top_indices = checkpoint['global_top_indices'].to(device)
    print(f"Resuming from passage index {start_idx}")

In [ ]:
chunk_size = 50000
print(f"Starting chunked exact search for {len(doc_ids)} passages...")
prof = LatencyProfiler()

with prof.time("bge_retrieval_total"):
    for i in range(start_idx, len(doc_ids), chunk_size):
        chunk_docs = passages[i : i + chunk_size]
        
        print(f"\nEncoding chunk {i} to {i + len(chunk_docs)}...")
        chunk_embeddings = model.encode(chunk_docs, batch_size=256, show_progress_bar=True, normalize_embeddings=True, convert_to_tensor=True)
        chunk_embeddings = chunk_embeddings.to(device)
        
        print("Calculating similarities and updating top-k...")
        similarities = torch.matmul(query_embeddings, chunk_embeddings.T)
        
        current_k = min(k, similarities.size(1))
        chunk_top_scores, chunk_top_indices = torch.topk(similarities, current_k, dim=1)
        chunk_top_indices += i
        
        combined_scores = torch.cat([global_top_scores, chunk_top_scores], dim=1)
        combined_indices = torch.cat([global_top_indices, chunk_top_indices], dim=1)
        
        new_global_top_scores, top_k_positions = torch.topk(combined_scores, k, dim=1)
        new_global_top_indices = torch.gather(combined_indices, 1, top_k_positions)
        
        global_top_scores = new_global_top_scores
        global_top_indices = new_global_top_indices
        
        del chunk_embeddings, similarities, chunk_top_scores, chunk_top_indices, combined_scores, combined_indices
        torch.cuda.empty_cache()
        
        torch.save({
            'last_chunk_end': i + len(chunk_docs),
            'global_top_scores': global_top_scores.cpu(),
            'global_top_indices': global_top_indices.cpu()
        }, checkpoint_path)
        print(f"✅ Finished chunk {i} to {i + len(chunk_docs)}. Checkpoint saved.")

print("✅ All chunks processed!")

In [ ]:
print("Formatting retrieval results...")
bge_run = {}
global_top_indices_cpu = global_top_indices.cpu().numpy()

for q_idx, qid in enumerate(query_ids):
    retrieved_doc_ids = [doc_ids[idx] for idx in global_top_indices_cpu[q_idx] if idx != -1]
    bge_run[qid] = retrieved_doc_ids

save_run_json(bge_run, "/workspace/results/bge_run.json")
print("✅ Retrieval complete and saved.")

In [ ]:
metrics = evaluate_run(bge_run, qrels)
print_metrics(metrics, system_name="BGE Dense Exact Baseline")

sanity_check_ndcg("bge", metrics["ndcg@10"])

print("\nLatency Profiling (Total Batch Time):")
prof.print_summary()

tracker = ExperimentTracker("/workspace/results/experiments.json")
tracker.log("BGE baseline", config={"model": "BAAI/bge-large-en-v1.5", "backend": "exact_search_chunked"}, metrics={
    "ndcg@10": metrics["ndcg@10"],
    "mrr@10": metrics["mrr@10"],
    "recall@100": metrics["recall@100"]
})
print("✅ Metrics logged to experiments.json")